In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import joblib

In [3]:
df = pd.read_csv("C:/Users/VISHAKA/Downloads/carbon_footprint_classification_dataset.csv")

df.head()

,product_id,product_name,product_type,material_composition,weight_kg,quantity,manufacturing_country,transport_mode,transport_distance_km,packaging_type,recycled_content_percent,renewable_energy_used_percent,manufacturing_energy_kwh,packaging_weight_kg,total_co2e_kg,carbon_class
0,P00001,Smart Headlight Unit,Automotive,Aluminum,34.76,5,Bangladesh,Ship,6463.4,NaN,29.3,18.9,1030.84,0.000,568.02,High
1,P00002,Compact Office Chair,Furniture,Wood,12.86,19,Germany,Ship,14259.0,Foam,0.8,62.5,25.86,1.229,77.74,Medium
2,P00003,Max Rain Jacket,Clothing,Polyester,1.09,9,Vietnam,Truck,844.3,Plastic,17.8,35.4,9.28,0.063,34.73,Low
3,P00004,Plus Perfume Bottle,Beauty,Plastic,1.15,23,Bangladesh,Truck,1491.4,Paper,15.6,0.0,7.74,0.086,51.43,Low
4,P00005,Classic Vacuum Cleaner,Home Appliances,Aluminum,45.00,39,India,Rail,2174.3,Foam,29.5,23.1,1876.47,4.140,593.79,High


In [4]:
print(df.shape)

print(df.info())

print(df.describe())

print(df.isnull().sum())

(5000, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   product_id                     5000 non-null   object 
 1   product_name                   5000 non-null   object 
 2   product_type                   5000 non-null   object 
 3   material_composition           5000 non-null   object 
 4   weight_kg                      5000 non-null   float64
 5   quantity                       5000 non-null   int64  
 6   manufacturing_country          5000 non-null   object 
 7   transport_mode                 5000 non-null   object 
 8   transport_distance_km          5000 non-null   float64
 9   packaging_type                 4696 non-null   object 
 10  recycled_content_percent       5000 non-null   float64
 11  renewable_energy_used_percent  5000 non-null   float64
 12  manufacturing_energy_kwh       5000 n

In [5]:
df["packaging_type"] = df["packaging_type"].fillna(
    df["packaging_type"].mode()[0]
)

In [6]:
df.isnull().sum()

product_id                       0
product_name                     0
product_type                     0
material_composition             0
weight_kg                        0
quantity                         0
manufacturing_country            0
transport_mode                   0
transport_distance_km            0
packaging_type                   0
recycled_content_percent         0
renewable_energy_used_percent    0
manufacturing_energy_kwh         0
packaging_weight_kg              0
total_co2e_kg                    0
carbon_class                     0
dtype: int64

In [7]:
df = df.drop(
    columns=[
        "product_id",
        "product_name"
    ]
)

In [8]:
X = df.drop("carbon_class", axis=1)

y = df["carbon_class"]

In [9]:
cat_cols = X.select_dtypes(include="object").columns

num_cols = X.select_dtypes(exclude="object").columns

print(cat_cols)

print(num_cols)

Index(['product_type', 'material_composition', 'manufacturing_country',
       'transport_mode', 'packaging_type'],
      dtype='object')
Index(['weight_kg', 'quantity', 'transport_distance_km',
       'recycled_content_percent', 'renewable_energy_used_percent',
       'manufacturing_energy_kwh', 'packaging_weight_kg', 'total_co2e_kg'],
      dtype='object')


In [10]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [17]:
model.fit(X_train, y_train)

C:\ ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
y_pred = model.predict(X_test)

In [19]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

Accuracy : 0.869


In [20]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        High       0.87      0.91      0.89       252
         Low       0.92      0.86      0.89       349
      Medium       0.83      0.85      0.84       399

    accuracy                           0.87      1000
   macro avg       0.87      0.87      0.87      1000
weighted avg       0.87      0.87      0.87      1000



In [21]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[230   0  22]
 [  0 301  48]
 [ 34  27 338]]


In [22]:
joblib.dump(model, "carbon_footprint_model.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [23]:
joblib.dump(list(y.unique()), "class_names.pkl")

['class_names.pkl']

In [24]:
import joblib

model = joblib.load("carbon_footprint_model.pkl")

In [25]:
new_data = pd.DataFrame({
    "product_type": ["Electronics"],
    "material_composition": ["Plastic"],
    "weight_kg": [2.5],
    "quantity": [10],
    "manufacturing_country": ["India"],
    "transport_mode": ["Road"],
    "transport_distance_km": [800],
    "packaging_type": ["Box"],
    "recycled_content_percent": [25],
    "renewable_energy_used_percent": [60],
    "manufacturing_energy_kwh": [90],
    "packaging_weight_kg": [0.5],
    "total_co2e_kg": [140]
})

prediction = model.predict(new_data)

print(prediction)

['Medium']
